# Atividade 1 · Como os dados da Quantum entram no banco do Q? — v9
**Arquitetura de Dados · MBA AI Engineering & Multi-Agents (FIAP)**

Boas-vindas à nossa primeira imersão prática! Hoje, vamos vivenciar um desafio real de arquitetura de dados na **Quantum Finance**. Nosso objetivo é construir o sistema de armazenamento de conhecimento e de memória para o agente virtual **Q**.

### O Cenário que vamos resolver:
A Quantum Finance nos entregou:
1. **59 manuais** no diretório `corpus/` (arquivos `.md` contendo metadados no cabeçalho como `titulo`, `area`, `tipo`, `status`, `data`). Destes, 40 são documentos oficiais vigentes e 19 são ruídos históricos (marketing antigo, rascunhos e FAQs defasados).
2. **24 interações de conversas passadas** de 8 clientes diferentes no arquivo `dados/conversas_seed.json`.

### O nosso desafio de arquitetura:
O chassi do agente **Q** já está implementado e pronto na bancada. Porém, sem uma infraestrutura de banco de dados, ele não consegue responder corretamente e não lembra das interações anteriores dos clientes. Nosso trabalho é **modelar e configurar as tabelas de banco de dados no pgvector (PostgreSQL)** para que o agente consiga:
- Localizar e citar apenas os manuais oficiais e vigentes.
- Lembrar do histórico de conversa exclusivo do cliente que está autenticado, com isolamento absoluto de segurança.
- Apagar de forma definitiva e permanente os dados de qualquer cliente que solicitar a exclusão de suas informações, em total conformidade com a LGPD.

### Como vamos nos organizar:
Em nosso grupo de estudos, podemos dividir as atenções de maneira colaborativa:
* **Foco no Design de Modelagem:** Vamos analisar quais metadados precisam virar colunas físicas nas tabelas e planejar as restrições e tipos de dados.
* **Foco em Segurança e Implementação:** Vamos estruturar as queries SQL para garantir que a busca semântica respeite os filtros de status e que a exclusão física (`DELETE`) limpe os dados completamente.
* **Garantia de Qualidade:** Vamos rodar o avaliador automático para medir nosso progresso, validar se o isolamento de privacidade está blindado e verificar se o agente consegue citar as fontes corretamente.

*Dica do Professor: Este notebook é um espaço de experimentação e tomada de decisão arquitetural. O código de partida que fornecemos é intencionalmente simples e vai falhar no avaliador no início. Nosso papel é analisar onde estão as lacunas e fazer as modificações necessárias para atingir o nível de produção!*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0 · Ambiente (Colab)

In [1]:
%%bash
# ── Célula 1 · Postgres + pgvector dentro da VM do Colab (~2–3 min; idempotente: se a VM reiniciar, rode de novo) ──
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq > /dev/null
apt-get install -y -qq postgresql postgresql-server-dev-all build-essential git > /dev/null
if ! ls /usr/lib/postgresql/*/lib/vector.so > /dev/null 2>&1; then
  rm -rf /tmp/pgvector
  git clone -q --branch v0.8.0 https://github.com/pgvector/pgvector.git /tmp/pgvector
  (cd /tmp/pgvector && make -s && make -s install)
fi
service postgresql start > /dev/null
sleep 2                                   # dá tempo ao socket do Postgres abrir antes de o Python conectar
su postgres -c "psql -qc \"ALTER USER postgres PASSWORD 'quantum';\""
su postgres -c "psql -qc 'CREATE EXTENSION IF NOT EXISTS vector;'"
su postgres -c "psql -Atc \"SELECT 'pgvector ' || extversion FROM pg_extension WHERE extname='vector';\""
pg_config --version

pgvector 0.8.0
PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Note: switching to '2627c5ff775ae6d7aef0c430121ccf857842d2f2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [2]:
# ── Célula 2 · Obter o kit (corpus, dados, chassi, avaliadores) ──

KIT_URL = "https://raw.githubusercontent.com/leandrommendes/vectordb-operacao-q/main/dist/operacao-q-v8.zip"

import os, sys, zipfile, urllib.request
raiz = next((c for c in (".", "operacao-q-v8", "..", "../..") if os.path.isdir(os.path.join(c, "kit_q"))), None)
if raiz is None:                              # kit ainda não está na VM
    if KIT_URL:
        urllib.request.urlretrieve(KIT_URL, "kit.zip")
    else:                                     # Opção B: upload manual do operacao-q-v8.zip
        from google.colab import files
        nome = list(files.upload().keys())[0]
        os.rename(nome, "kit.zip")
    zipfile.ZipFile("kit.zip").extractall(".")
    raiz = "operacao-q-v8"
os.chdir(raiz)
sys.path.insert(0, os.getcwd())
print("📁 kit em", os.getcwd(), "·", len(os.listdir("corpus")), "arquivos no corpus")

📁 kit em /content/operacao-q-v8 · 60 arquivos no corpus


In [3]:
%pip -q install "psycopg[binary]" pgvector sentence-transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 15.5 MB/s eta 0:00:00


In [4]:
import sys
# Ensure psycopg is installed and available, as ModuleNotFoundError occurred.
# This might be redundant if the pip install cell was run, but ensures availability.
try:
    import psycopg
except ModuleNotFoundError:
    !{sys.executable} -m pip install -q "psycopg[binary]"
    import psycopg

from kit_q import conectar, emb, explain, chunks_do_corpus, load_conversas
conn = conectar()
print("dimensões do embedding:", emb(["teste"]).shape[1])   # 384 — o número que define a coluna vector(384)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

dimensões do embedding: 384


## 1 · Reconhecendo e Explorando os Nossos Dados
Antes de começarmos a modelar as tabelas físicas no banco, vamos olhar com calma o que a Quantum Finance nos entregou. Que metadados existem no cabeçalho de cada documento do corpus? O que diferencia, por exemplo, um manual de políticas oficial de um rascunho de marketing? E o que as interações anteriores de conversas trazem de informação? Rodem a célula abaixo para explorar as estruturas!



---


⬇ **RESPOSTAS DO GRUPO** ⬇

Os metadados do cabeçalho dos documentos do corpus são: empresa (geralmente Quantum Finance), area (área da empresa que classifica o documento), tipo (tipo do documento), data (data que o documento foi criado ou atualizado), titulo (o que define o documento) e status (uma classificação extra do documento talvez sobre a confidencialidade ou relevância).

O que diferencia um conteúdo oficial de um rascunho de marketing é o metadado status presente no cabeçalho. Para documentos oficiais o status é igual a "oficial". No caso dos rascunhos de marketing, o status entra como "ruído".

As interações anteriores prepara e monta o ambiente aqui no Colab, depois são baixados todos os arquivos para alimentação dos dados no bando Postgres onde na sequência, todos os materiais são convertidos em vetores via pgvector (chunks são criados). Assim elas trazem de informação o resultado destes processos, deixando nosso banco de dados vetorizado e preparado para consultas e manutenções.

In [5]:
chunks = chunks_do_corpus()          # 244 chunks (um por parágrafo), cada um com TODOS os campos do front-matter
print(len(chunks), "chunks ·", len({c["doc_id"] for c in chunks}), "documentos")
print("campos disponíveis por chunk:", list(chunks[0].keys()))
for c in chunks[:2]:
    print({k: (v[:60] + "…" if isinstance(v, str) and len(v) > 60 else v) for k, v in c.items()})

from collections import Counter
print("\nstatus:", Counter(c["status"] for c in chunks))
print("tipo:", Counter(c["tipo"] for c in chunks))

conversas = load_conversas()
print("\n", len(conversas), "conversas ·", sorted({c["cliente_id"] for c in conversas}))
print(conversas[0])

244 chunks · 59 documentos
campos disponíveis por chunk: ['id', 'doc_id', 'n_chunk', 'titulo', 'area', 'tipo', 'status', 'data', 'empresa', 'texto']
{'id': 'ata-comite-produtos-2025-11#0', 'doc_id': 'ata-comite-produtos-2025-11', 'n_chunk': 0, 'titulo': 'Ata do Comitê de Produtos — Novembro/2025', 'area': 'Diretoria', 'tipo': 'ata', 'status': 'oficial', 'data': '2025-11-18', 'empresa': 'Quantum Finance', 'texto': 'Ata do Comitê de Produtos — Novembro/2025 — # Ata do Comitê …'}
{'id': 'ata-comite-produtos-2025-11#1', 'doc_id': 'ata-comite-produtos-2025-11', 'n_chunk': 1, 'titulo': 'Ata do Comitê de Produtos — Novembro/2025', 'area': 'Diretoria', 'tipo': 'ata', 'status': 'oficial', 'data': '2025-11-18', 'empresa': 'Quantum Finance', 'texto': 'Ata do Comitê de Produtos — Novembro/2025 — Presentes: CPO (…'}

status: Counter({'oficial': 199, 'ruido': 45})
tipo: Counter({'politica': 85, 'manual': 48, 'marketing': 24, 'ata': 16, 'faq': 14, 'rascunho': 12, 'comunicado': 10, 'faq-antigo': 9, 't

## 2 · Decisão D1 — Design do Schema: Quais metadados virão colunas físicas?
No ponto de partida padrão, as tabelas foram criadas contendo apenas as colunas de `texto` e `embedding`. Isso funciona para buscas básicas, mas pensem comigo: como faremos para filtrar apenas os documentos oficiais? Como o agente Q saberá qual documento citar como fonte das tarifas? E na tabela de memória, como vamos isolar as conversações de cada cliente e saber qual delas é a mais recente?

**Perguntas para Reflexão de Arquitetura:**
1. Quais campos do cabeçalho dos documentos do corpus precisam se transformar em colunas físicas na tabela `conhecimento` para permitir filtros rápidos (B-tree) ou junções?
2. Quais tipos de dados (date, text, NOT NULL) são adequados para cada uma dessas colunas?
3. E na tabela `memoria`, como podemos desenhar as restrições para garantir que os dados de cada cliente fiquem estritamente isolados e possam ser apagados individualmente de acordo com a LGPD?

---
⬇ **RESPOSTAS DO GRUPO** ⬇

**1.**
A tabela `conhecimento` (Base Estática do Agente) armazena os fragmentos fatiados (chunks) dos manuais de procedimentos internos. Respondendo então quais metadados virarão colunas físicas e por quê:

* `tipo`: É o metadado mais crítico para a precisão do agente. Como a base contém manuais antigos e rascunhos que geram ruído, ter o tipo como coluna física indexável permite aplicar o filtro estruturado `WHERE tipo = 'oficial'`. Isso impede o agente de recuperar regras de tarifas defasadas.

* `doc_id`: Identifica de qual manual o trecho de texto se originou (ex: `prod-tabela-tarifas` ou `comp-politica-lgpd`). É fundamental para que o Agente Q consiga realizar junções ou exibir as citações e fontes exatas no chat da aplicação.

* `titulo`: Armazena o título amigável do documento. Ele deve ser uma coluna física para que o agente exiba o nome do documento citado sem precisar recalcular ou carregar dados externos.

* `data`: Permite realizar filtros e ordenações por recência. Caso existam dois documentos oficiais parecidos, a coluna física de data ajuda a priorizar o mais recente.

* `area`: Embora opcional em queries básicas, é útil como coluna física para filtros rápidos de departamento (ex: buscar apenas manuais da área de `cred` ou `rh`), otimizando o tempo de resposta.

**2.**
DDL Recomendado e Tipos de Dados:

```
CREATE TABLE conhecimento (
    id text PRIMARY KEY,
    doc_id text NOT NULL,
    titulo text NOT NULL,
    tipo text NOT NULL,
    data date NOT NULL,
    texto text NOT NULL,
    embedding vector(384) NOT NULL
);
```

**`id` (`text PRIMARY KEY`):** Identificador único de cada linha (geralmente gerado na ingestão como `doc_id#chunk_index`), garantindo a integridade dos dados e indexação primária rápida.

**`doc_id` / `titulo` / `tipo` / `texto` (`text NOT NULL`):** Usar o tipo `text` com a restrição `NOT NULL` assegura que nenhum dado essencial para filtragem semântica ou geração de citações fique em branco.

**`data` (`date NOT NULL`):** O tipo `date` nativo é ideal. Ele permite que o banco de dados processe buscas temporais (como "políticas criadas após 2026") de forma direta e otimizada por índices B-tree, evitando a sobrecarga e lentidão de parsing de textos em formato de data no runtime.

**`embedding` (`vector(384) NOT NULL`):** Tipo `vector(384)` rígido. É o tipo vetorial do pgvector configurado estritamente com **384 dimensões** para comportar os vetores gerados pelo modelo de inteligência artificial de faturamento local (*paraphrase-multilingual-MiniLM-L12-v2*).

**3.** Acreditamos nestes ajustes abaixo para a tabela `memoria` (Base Dinâmica de Interações), garantindo segurança e integridade dos dados armazenados e pesquisados:

Esta tabela gerencia o histórico dinâmico das conversas e preferências do usuário em tempo real, exigindo blindagem absoluta de acesso e conformidade legal.

DDL Recomendado para `memoria`:

```
CREATE TABLE memoria (
    id bigserial PRIMARY KEY,
    cliente_id text NOT NULL,
    texto text NOT NULL,
    embedding vector(384) NOT NULL,
    criado_em timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);
```

**Estratégia de Isolamento, Recência e LGPD:**

* **Garantia de Isolamento:** O isolamento de dados permanece totalmente seguro no nível da query. Toda busca semântica por preferências ou histórico de conversas executada pelo agente continuará aplicando obrigatoriamente o filtro na cláusula `WHERE`:
  ```
  SELECT texto FROM memoria
  WHERE cliente_id = :id_cliente_autenticado
  ORDER BY embedding <=> :q_vector
  LIMIT 3;
  ```
  Isso assegura que, em tempo de execução, o banco de dados separe de forma lógica e física as memórias de cada cliente.

* **Ordenação por Recência (`criado_em`):** A inclusão da coluna `criado_em` com tipo `timestamp with time zone` (e valor padrão `CURRENT_TIMESTAMP`) resolve o problema da linha temporal. Quando o agente consulta o histórico de preferências do usuário, podemos usar uma ordenação combinada (`ORDER BY criado_em DESC`), permitindo que o Agente Q saiba o que foi acordado ou conversado na interação mais recente de forma extremamente veloz.

* **Conformidade com a LGPD (Direito ao Esquecimento):**
Quando um cliente (como a Ana) solicita a exclusão de seus dados de acordo com a LGPD, o sistema deve disparar um comando *`DELETE` físico real (hard delete)* na tabela memoria.
  * *O risco do Soft Delete:* Utilizar apenas um marcador lógico (como uma flag `ativo = false`) esconde o registro nas buscas, mas mantém os dados do cliente e as conversas salvas no servidor, o que viola diretamente os requisitos de conformidade de eliminação permanente de dados exigidos pela LGPD.

  Quando um usuário solicitar a remoção de seus dados, bastará rodar:
  
  `DELETE FROM memoria WHERE cliente_id = :id;`
  
  Isso removerá permanentemente todos os registros de interações e embeddings associados àquele identificador, atendendo perfeitamente à conformidade legal exigida pela LGPD (hard delete).

* **Otimização com Índice:** Como a busca e a exclusão por `cliente_id` serão operações bastante frequentes, criar um índice B-tree nessa coluna é uma prática recomendada para evitar buscas lineares (*sequential scans*) na tabela:

  `CREATE INDEX idx_memoria_cliente ON memoria(cliente_id);`


In [6]:
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS conhecimento; DROP TABLE IF EXISTS memoria;")
    # ======================= D1 · DESIGN DO SCHEMA =======================
    # Time, atenção aqui!
    # O ponto de partida abaixo funciona para buscas semânticas puras, mas ele é "ingênuo".
    # Pensem no mundo real da Quantum:
    #   1. Se o Q encontrar a resposta, como ele cita a fonte de forma AUDITÁVEL se não guardamos
    #      o identificador do documento de origem? (auditoria e rastreabilidade)
    #   2. Como ignorar rascunhos, marketing e FAQs antigos se o banco não sabe o STATUS do documento?
    #   3. Qual o impacto de injetar no prompt uma regra revogada se não existe coluna de VIGÊNCIA
    #      (a data de publicação)? Um manual sem vigência é uma tarifa errada esperando para acontecer.
    #
    # Desafio do grupo: modelem as colunas extras que dão ao agente filtro e rastreabilidade —
    # e escolham os TIPOS com cuidado (uma data guardada como text não ordena nem compara direito).

    # ESTA É VERSÃO PADRÃO DO PROFESSOR:
    # cur.execute('''
    #     CREATE TABLE conhecimento (
    #         id        text PRIMARY KEY,
    #         texto     text,
    #         embedding vector(384)
    #         -- Dica do Prof: que colunas de metadado (doc_id, titulo, status, data…) precisam existir aqui?
    #         -- Lembrem-se de mapear os tipos corretamente (text, date, NOT NULL…).
    #     )''')
    # ESTA É A NOSSA VERSÃO PARA O EXERCÍCIO:
    cur.execute('''
        CREATE TABLE conhecimento (
            id text PRIMARY KEY,
            doc_id text NOT NULL,
            titulo text NOT NULL,
            tipo text NOT NULL,
            data date NOT NULL,
            texto text NOT NULL,
            status text,
            embedding vector(384) NOT NULL
        )''')

    # Na memória, o schema É a política de segurança:
    #   · cliente_id aceitando NULL significa que existe memória "de ninguém" — e memória de ninguém
    #     é memória de todo mundo. O isolamento (D3) e a exclusão LGPD dependem desta coluna ser obrigatória.
    #   · sem timestamp não há recência (D4), não há política de retenção e não há como responder
    #     a uma auditoria: "quando o agente soube disso?". Memória temporal exige o QUANDO.

    # ESTA É VERSÃO PADRÃO DO PROFESSOR:
    # cur.execute('''
    #     CREATE TABLE memoria (
    #         id         bigserial PRIMARY KEY,
    #         cliente_id text,                  -- 🚨 pode ser NULL? o isolamento e o esquecer() dependem disso
    #         texto      text,
    #         embedding  vector(384)
    #         -- Dica do Prof: e o QUANDO (timestamp) para recência, retenção e conformidade?
    #     )''')

    # ESTA É A NOSSA VERSÃO PARA O EXERCÍCIO:
    cur.execute('''
        CREATE TABLE memoria (
           id bigserial PRIMARY KEY,
           cliente_id text NOT NULL,
           texto text NOT NULL,
           embedding vector(384) NOT NULL,
           origem text,
           criado_em timestamp with time zone
        )''')

    # =====================================================================
    cur.execute("SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_name IN ('conhecimento','memoria') ORDER BY 1, ordinal_position")
    for r in cur.fetchall(): print(r)
    # RESPOSTA PRELIMINAR DO PROFESSOR:
    # ('conhecimento', 'id', 'text')
    # ('conhecimento', 'texto', 'text')
    # ('conhecimento', 'embedding', 'USER-DEFINED')
    # ('memoria', 'id', 'bigint')
    # ('memoria', 'cliente_id', 'text')
    # ('memoria', 'texto', 'text')
    # ('memoria', 'embedding', 'USER-DEFINED')


('conhecimento', 'id', 'text')
('conhecimento', 'doc_id', 'text')
('conhecimento', 'titulo', 'text')
('conhecimento', 'tipo', 'text')
('conhecimento', 'data', 'date')
('conhecimento', 'texto', 'text')
('conhecimento', 'status', 'text')
('conhecimento', 'embedding', 'USER-DEFINED')
('memoria', 'id', 'bigint')
('memoria', 'cliente_id', 'text')
('memoria', 'texto', 'text')
('memoria', 'embedding', 'USER-DEFINED')
('memoria', 'origem', 'text')
('memoria', 'criado_em', 'timestamp with time zone')


## 3 · Processo de Ingestão do Conhecimento
A esteira de processamento (que realiza as etapas de leitura, fatiamento, vetorização e inserção) já está pré-configurada para nós. O papel do nosso grupo aqui é decidir e programar o **mapeamento**: quais campos do dicionário de metadados do chunk devem ser associados a quais colunas físicas da nossa tabela (isso precisa bater exatamente com o schema físico que desenhamos na etapa D1!).

In [7]:
# 1. Limpa a tabela para garantir que não haverá dados corrompidos
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE conhecimento")
conn.commit()

from kit_q.ingestao import ingerir_conhecimento

def mapear(chunk):
    # ======================= D1 · MAPEAMENTO =======================
    # Tudo que vocês criaram como coluna no D1 precisa ser preenchido aqui — e só aqui.
    # O chunk já traz todos os campos do cabeçalho (veja chunks[0].keys()); o que não for mapeado
    # simplesmente não existe para o banco, e portanto não existe para o Q.
    # return {"id": chunk["id"], "texto": chunk["texto"]}      # 🚨 só isso? e o doc_id, status, data, titulo?

    # SEGUE O NOSSO MAPEAMENTO CONFORME OS NOVOS CAMPOS DA TABELA CONHECIMENTO:
    # Extraímos o doc_id dinamicamente dividindo o ID do chunk no caractere '#'
    # Exemplo: 'prod-tabela-tarifas#0' vira 'prod-tabela-tarifas'
    #doc_id = chunk["id"].split("#")
    doc_id_string = chunk["id"].split("#")[0]
    return {
        "id": chunk["id"],
        "doc_id": doc_id_string,
        # Trata flexibilidade de acentuação do front-matter original
        "titulo": chunk.get("titulo") or chunk.get("título"),
        "tipo": chunk["tipo"],
        "data": chunk["data"],
        "texto": chunk["texto"],
        "status": chunk["status"]
    }
    # ==============================================================

ingerir_conhecimento(conn, mapear)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM conhecimento");
    # print("linhas:", cur.fetchone()[0])
    print("Linhas totais inseridas na tabela conhecimento:", cur.fetchone())

conn.commit()  # Salva fisicamente no Postgres
print("Ingestão concluída com sucesso!")

# ==============================================================================
# INSPEÇÃO DIRETA DA TABELA CONHECIMENTO
# ==============================================================================
with conn.cursor() as cur:
    # Selecionamos apenas as colunas de texto/metadados para não poluir a tela com vetores gigantes
    cur.execute("SELECT id, doc_id, titulo, tipo, data, status, texto FROM conhecimento LIMIT 10")
    linhas = cur.fetchall()

    print(f"Total de registros encontrados (máx 50): {len(linhas)}")
    print("=" * 80)

    if len(linhas) == 0:
        print("🚨 A TABELA ESTÁ COMPLETAMENTE VAZIA!")
        print("Certifique-se de que rodou a ingestão e executou 'conn.commit()' para salvar os dados.")
    else:
        for i, r in enumerate(linhas, 1):
            print(f"Registro #{i}")
            print(f"ID: {r[0]}")
            print(f"Doc ID (Nome do Arquivo): {r[1]}")
            print(f"Título: {r[2]}")
            print(f"Tipo (Status): {r[3]}")
            print(f"Data: {r[4]}")
            print(f"Status: {r[5]}")
            print(f"Texto (Resumo): {r[6][:120]}...")
            print("-" * 50)

✅ 244 chunks inseridos em conhecimento (23.8s) · colunas: ['id', 'doc_id', 'titulo', 'tipo', 'data', 'texto', 'status', 'embedding']
Linhas totais inseridas na tabela conhecimento: (244,)
Ingestão concluída com sucesso!
Total de registros encontrados (máx 50): 10
Registro #1
ID: ata-comite-produtos-2025-11#0
Doc ID (Nome do Arquivo): ata-comite-produtos-2025-11
Título: Ata do Comitê de Produtos — Novembro/2025
Tipo (Status): ata
Data: 2025-11-18
Status: oficial
Texto (Resumo): Ata do Comitê de Produtos — Novembro/2025 — # Ata do Comitê de Produtos — Novembro/2025...
--------------------------------------------------
Registro #2
ID: ata-comite-produtos-2025-11#1
Doc ID (Nome do Arquivo): ata-comite-produtos-2025-11
Título: Ata do Comitê de Produtos — Novembro/2025
Tipo (Status): ata
Data: 2025-11-18
Status: oficial
Texto (Resumo): Ata do Comitê de Produtos — Novembro/2025 — Presentes: CPO (presidência), CFO, Head de Crédito, Head de Investimentos, r...
----------------------------------

## 4 · Decisão D2 — Consulta de Conhecimento com Filtragem Ativa
O contrato que o nosso avaliador automático espera é: `buscar_conhecimento(query, filtros=None, k=4)`. Ele deve nos retornar uma lista de dicionários contendo pelo menos as chaves `doc_id`, `titulo` e `texto`, ordenadas pela maior proximidade semântica (menor distância de cosseno).

Atualmente, o ponto de partida do código ignora completamente o parâmetro `filtros`. Vamos testar isso: se fizermos uma busca sem filtros pela frase "quanto custa sacar?", o agente vai ler as tarifas corretas da Tabela de Tarifas atual ou vai acabar confundindo-se com o FAQ de marketing antigo? Vamos ajustar o código para que o Postgres execute os filtros do WHERE dinamicamente!

In [ ]:
def buscar_conhecimento(query, filtros=None, k=4):
    # v = emb([query])[0]
    # ======================= D2 · RETRIEVAL COM INTELIGÊNCIA DE NEGÓCIO =======================
    # Olá, time de desenvolvimento! Aqui a engenharia de busca se conecta com as regras da Quantum.
    # Se o chamador passa um dicionário de filtros (ex.: {"status": "oficial"}), a query SQL
    # NÃO pode ignorá-lo e fazer um SELECT cego na tabela inteira, torcendo para o vetor acertar.
    #
    # Objetivo de vocês: montar a query dinamicamente — se houver filtros, o Postgres aplica a
    # cláusula WHERE junto com a ordenação vetorial (é o banco que filtra, não o Python depois).
    # Pensem também na vigência: um filtro por data mínima (data >= …) evita citar regra revogada.
    # E não esqueçam de devolver doc_id e titulo: sem eles o Q não sabe de onde leu.
    # sql = "SELECT id, texto FROM conhecimento ORDER BY embedding <=> %s LIMIT %s"   # 🚨 e o WHERE? e o doc_id/titulo?
    # params = (v, k)
    # ==========================================================================================
    # with conn.cursor() as cur:
    #     cur.execute(sql, params)
    #     return [{"doc_id": r[0].split("#")[0], "titulo": None, "texto": r[1]} for r in cur.fetchall()]

    filtros = filtros or {}
    vetor = emb([query])[0]
    vetor_str = "[" + ",".join(
        format(float(valor), ".10g")
        for valor in vetor
    ) + "]"
    sql = """
        SELECT
            id,
            doc_id,
            titulo,
            status,
            data,
            texto,
            embedding <=> %s::vector AS distancia
        FROM conhecimento
    """
    condicoes = []
    params = [vetor_str]
    # sem filtro de status: exclui ruido; com filtro: o chamador manda (ex. oficial)
    if not filtros.get("status"):
        condicoes.append("LOWER(TRIM(status)) <> 'ruido'")
    if filtros.get("status"):
        condicoes.append("LOWER(TRIM(status)) = LOWER(TRIM(%s))")
        params.append(str(filtros["status"]))
    if filtros.get("tipo"):
        condicoes.append("LOWER(TRIM(tipo)) = LOWER(TRIM(%s))")
        params.append(str(filtros["tipo"]))
    if filtros.get("data_min"):
        condicoes.append("data >= %s::date")
        params.append(filtros["data_min"])
    elif isinstance(filtros.get("data"), dict):
        if filtros["data"].get("gte"):
            condicoes.append("data >= %s::date")
            params.append(filtros["data"]["gte"])
    elif filtros.get("data"):
        condicoes.append("data = %s::date")
        params.append(filtros["data"])
    if filtros.get("doc_id"):
        condicoes.append("LOWER(TRIM(doc_id)) = LOWER(TRIM(%s))")
        params.append(str(filtros["doc_id"]))
    if condicoes:
        sql += " WHERE " + " AND ".join(condicoes)
    sql += " ORDER BY distancia LIMIT %s"
    params.append(int(k))
    with conn.cursor() as cur:
        cur.execute(sql, tuple(params))
        registros = cur.fetchall()
    return [
        {
            "id": registro[0],
            "doc_id": registro[1],
            "titulo": registro[2],
            "status": registro[3],
            "data": registro[4],
            "texto": registro[5],
            "distancia": float(registro[6])
        }
        for registro in registros
    ]

## EXERCICIO ORIGINAL DO PROFESSOR
# print("\n\nEXERCICIO ORIGINAL DO PROFESSOR\n")
# for r in buscar_conhecimento("quanto custa sacar?", k=3):
#     print(r["doc_id"], "·", r["texto"][:150])
# print("\ncom filtro (status = oficial):")
# for r in buscar_conhecimento("quanto custa sacar?", filtros={"status": "oficial"}, k=3):
#     print(r["doc_id"], "·", r["texto"][:150])
# print("\nplano de execução:")
# explain(conn, "SELECT id FROM conhecimento ORDER BY embedding <=> %s LIMIT 3", (v := emb(["saque"])[0],))

# RESPOSTA ATUAL DO PROFESSOR QUANDO RODAMOS A VERSÃO ORIGINAL:
# prod-tabela-tarifas · Tabela de Tarifas — Pessoa Física — Manutenção de conta: isenta. Pix: gratuito para pessoa física. TED: gratuita pelo app.
# DOC: descontinuado. Saque e
# ti-faq-app · FAQ do App Quantum — Suporte Nível 1 — **Quanto custa o saque?** O saque na rede Banco24Horas custa **R$ 6,90 por operação**,
# com 2 saques gratuitos p
# comunicado-antigo-tarifas-2024 · Comunicado arquivado: tabela de tarifas 2024 — Em fevereiro de 2024, o saque na rede Banco24Horas custava R$ 6,50 com apenas 1 saque
# gratuito por mês,
#
# com filtro (status = oficial):
# prod-tabela-tarifas · Tabela de Tarifas — Pessoa Física — Manutenção de conta: isenta. Pix: gratuito para pessoa física. TED: gratuita pelo app.
# DOC: descontinuado. Saque e
# ti-faq-app · FAQ do App Quantum — Suporte Nível 1 — **Quanto custa o saque?** O saque na rede Banco24Horas custa **R$ 6,90 por operação**,
# com 2 saques gratuitos p
# comunicado-antigo-tarifas-2024 · Comunicado arquivado: tabela de tarifas 2024 — Em fevereiro de 2024, o saque na rede Banco24Horas custava R$ 6,50 com apenas 1 saque
# gratuito por mês,
#
# plano de execução:
#    ->  Seq Scan on conhecimento
# ['Limit',
#  '  ->  Sort',
#  "        Sort Key: ((embedding <=> '[0.005932082,0.050468117,-0.051851645,0.021228915,-0.03267979,-0.039447986,0.1268404,0.03793605,0.004905199,-0.019042546,0.025953524,-0.08345301,0.024886014,-0.017689321,-0.014671685,-0.018040711,-0.0019083191,0.03137811,0.010086734,-0.008232487,0.067635536,-0.013655133,-0.02153172,-0.016574262,-0.022102619,-0.045576006,0.006222678,0.0002579101,-0.03418484,-0.10655139,0.010332363,-0.040286724,0.03913175,0.047670625,-0.0098206615,0.052813455,0.004577609,0.007973723,0.012828839,0.034598436,0.007640795,0.01621976,-0.022603374,0.056105647,0.020131363,0.021498224,-0.0040884246,0.02600486,-0.0050255954,0.014997438,-0.0064840387,-0.0006658872,-0.05963078,-0.033078913,0.018393435,0.013323099,0.029100044,0.014164307,-0.01784444,0.011798822,0.012982239,0.057385083,-0.18375796,0.040894702,-0.04067098,-0.08493317,0.022828827,-0.0091482485,-0.07273707,0.09457943,-0.018337008,-0.030689502,0.004429979,0.04307873,0.03856938,-0.034697734,0.011434347,0.046529908,-0.03652271,-0.041999597,0.041367475,0.036637664,-0.004038841,0.002954834,-0.025690971,0.017199438,0.004129241,0.024001336,0.091701776,-0.04553509,0.050052516,-0.055413518,0.06066144,0.02037027,0.057027,-0.018036516,0.012500092,-0.014110726,0.049087644,0.550491,-0.0037606854,0.0074100113,0.02567606,0.027557489,-0.03356273,-0.026386807,0.040668726,-0.030530697,-0.014661358,-0.00030746817,-0.053619787,0.022079017,-0.022921942,0.0020098963,-0.013297442,0.0116024595,-0.009936436,0.016110914,-0.016115867,0.008428393,-0.020749819,-0.007487553,0.023708683,0.03857632,0.0015536654,-0.15256007,0.019863352,0.060475167,-0.021762531,0.020201627,0.00013210955,0.021971514,-0.024052173,0.0017528664,-0.046202388,0.0017878647,0.021041805,0.07079502,0.01142364,-0.1078947,-0.0608886,0.068604864,0.01885636,0.006589091,0.042481195,-0.04959632,0.0007578128,-0.07991858,-0.0007304709,0.06182292,-0.0103972275,-0.048042122,0.0027198216,0.0153687745,0.03609893,-0.03544409,-0.022972587,0.01613819,0.03657649,0.06456654,-0.0333836,0.026546905,0.009387225,0.0005436323,-0.029008092,-0.024754772,-0.041494578,0.042154416,0.06968214,0.032633234,0.04257097,-0.000761279,-0.013217142,0.022070596,-0.0045160516,-0.010562025,-0.0115223015,0.01782026,-0.032440044,0.044415098,0.016948685,0.017424073,-0.031195052,0.07563217,0.0060041696,0.023449272,-0.026251193,0.0061004027,0.0434327,-0.06258694,-0.04764532,-0.019577203,-6.6456923e-06,-0.020098623,-0.017127516,-0.04912841,-0.0051278365,-0.03560435,-0.021622766,0.016455026,-0.08007528,-0.027026078,-0.015036742,-0.051278174,0.09118027,-0.04452362,0.011725525,-0.013666957,-0.020708121,0.0301167,0.03336081,-0.022665951,-0.043020796,0.011238166,0.030856084,-0.011497565,0.020985141,-0.0045775245,-0.06419825,0.036071785,0.016905839,-0.06987076,0.013279543,-0.1772303,0.08391418,-0.04463964,-0.034734752,0.053582314,-0.0059560896,-0.02047465,0.016552197,0.005361344,-0.0012765685,-0.028583923,-0.08600723,-0.070811115,0.034360558,-0.030085836,-0.00837533,0.0077404846,-0.039569665,-0.0152526265,0.014381068,0.061620712,-0.032285087,0.008359103,0.03267536,0.10213757,-0.031071903,0.11238645,-0.017862894,-0.007910784,-0.050495647,0.00737123,0.012994433,-0.025530221,-0.009206365,0.014025963,-0.0078504495,0.0088657215,0.0658406,-0.014225886,-0.09020988,0.0023548007,0.028098524,-0.01170574,-0.013027998,0.09676433,0.037456937,0.0213439,-0.03366031,-0.003855827,0.0014171096,-0.026478868,0.045455407,-0.007069853,0.009010347,-0.044714652,0.021878403,0.049937997,-0.011250083,-0.07533622,-0.017728336,0.031958222,0.030576942,0.017792089,-0.02942399,-0.020356495,-0.0010338208,-0.0010183469,-0.01868754,0.033830836,-0.0044428734,0.0017438116,0.007862501,-0.034211263,-0.007893743,0.04491008,-0.016478881,0.04694616,-0.034468997,-0.07706694,0.0059228283,-0.034991626,-0.04803752,-0.039327547,-0.05814787,0.047339503,0.08201452,0.0019737114,-0.05584188,-0.008310685,0.031678215,-0.019990424,0.022840269,-0.0008551263,-0.002084027,0.017116718,0.028536001,-0.23245451,0.011798859,-0.03154185,-0.109511085,0.07518915,0.040346604,-0.0154350335,-0.037263438,-0.055619985,0.016654272,0.011185506,-0.044295892,0.035955656,-0.06311592,-0.0017180414,-0.0045341044,0.036156476,-0.013124169,0.01702656,-0.025766337,-0.06770913,0.04202098,0.05407217,0.031422064,-0.04109594,-0.045229215,-0.009879195,-0.03770103,-0.06784599,0.029473718,-0.012787412,0.013575626,0.004340731,-0.053135566,-0.0150843905,-0.010605945,0.060192674,0.029470528,0.019569116,0.047167882,0.04321871,0.07917968,-0.012153446,0.03566548,0.05352448,-0.0028824843,0.03959411,-0.047109183,-0.014924777,-0.012941108,0.026427874,-0.0008836811,0.023014162,0.02952291,0.033380255,-0.010821642,-0.032428093,-0.009839572,-0.027233433,-0.033124406,0.049437456,0.084675066,-0.043996904,-0.03169384,0.037646532]'::vector))",
#  '        ->  Seq Scan on conhecimento']

# ==============================================================================
# SCRIPT DE DIAGNÓSTICO: VAMOS DESCOBRIR O QUE ESTÁ GRAVADO NO BANCO!
# ==============================================================================
with conn.cursor() as cur:
    print("=== 1. Valores e contagens na coluna TIPO (status) ===")
    cur.execute("SELECT tipo, count(*) FROM conhecimento GROUP BY tipo")
    valores_tipo = cur.fetchall()
    if not valores_tipo:
        print("🚨 NENHUM DADO ENCONTRADO NA COLUNA 'TIPO'!")
    for r in valores_tipo:
        print(f"   - Valor gravado: '{r}' | Contagem: {r[1]} linhas")

    print("\n=== 2. Formato dos dados na coluna DOC_ID ===")
    cur.execute("SELECT DISTINCT doc_id FROM conhecimento LIMIT 5")
    valores_doc = cur.fetchall()
    for r in valores_doc:
        print(f"   - doc_id gravado: {repr(r)} (tipo: {type(r).__name__})")

    print("\n=== 3. Janela temporal na coluna DATA ===")
    cur.execute("SELECT MIN(data), MAX(data) FROM conhecimento")
    datas = cur.fetchone()
    print(f"   - Data mais antiga: {datas} | Data mais recente: {datas[1]}")

    print("\n=== 4. Segmentação de TIPO na base de dados ===")
    cur.execute("SELECT tipo, COUNT(*) FROM conhecimento GROUP BY tipo ORDER BY tipo")
    for linha in cur.fetchall():
        print(linha)
    print("\n\n")

# ============================================================
# EXEMPLO 1 — Busca com filtro por tipo
# ============================================================
print("BUSCA 1 — Tarifa de saque")
print("Filtro: tipo = tabela\n")
resultados = buscar_conhecimento(
    query="Qual é a tarifa de saque?",
    filtros={"tipo": "tabela"},
    k=3
)
if resultados:
    for r in resultados:
        print("Documento:", r["doc_id"])
        print("Título:", r["titulo"])
        print("Texto:", r["texto"][:300])
        print("Distância:", r.get("distancia"))
        print("-" * 80)
else:
    print("Nenhum resultado encontrado.")

# ============================================================
# EXEMPLO 2 — Busca com tipo e data mínima
# ============================================================
print("\nBUSCA 2 — Rentabilidade do CDB Quantum")
print("Filtros: tipo = ata e data >= 2026-01-01\n")
resultados = buscar_conhecimento(
    query="Qual é a Rentabilidade do CDB Quantum?",
    filtros={
        "tipo": "ata",
        "data_min": "2026-01-01"
    },
    k=3
)
if resultados:
    for r in resultados:
        print("Documento:", r["doc_id"])
        print("Título:", r["titulo"])
        print("Texto:", r["texto"][:300])
        print("Distância:", r.get("distancia"))
        print("-" * 80)
else:
    print("Nenhum resultado encontrado.")

# ============================================================
# EXEMPLO 3 — Busca por documento específico
# ============================================================
print("\nBUSCA 3 — Cartão Black")
print("Filtro: doc_id = mkt-blog-cartao-black\n")
resultados = buscar_conhecimento(
    query="Gostaria de saber mais do Cartão Quantum Black.",
    filtros={
        "doc_id": "mkt-blog-cartao-black"
    },
    k=2
)
if resultados:
    for r in resultados:
        print("Documento:", r["doc_id"])
        print("Título:", r["titulo"])
        print("Texto:", r["texto"][:300])
        print("Distância:", r.get("distancia"))
        print("-" * 80)
else:
    print("Nenhum resultado encontrado.")
    print("PROFESSOR: Aqui não vai retornar porque o documento é do tipo RUIDO, e no busca_conhecimento restringimos apenas para tipo OFICIAL.")

# ============================================================
# EXEMPLO 4 - Busca por Tarifa de Saque
# ============================================================
print("\nBUSCA 4 - Tarifa de Saque (usada mais no final do exercicio)")
print("Filtro: status = oficial\n")
resultados = buscar_conhecimento(
    query="Qual é a tarifa de saque?",
    filtros={
        "status": "oficial"
    },
    k=2
)
if resultados:
    for r in resultados:
        print("Documento:", r["doc_id"])
        print("Título:", r["titulo"])
        print("Data:", r["data"])
        print("Texto:", r["texto"][:300])
        print("Distância:", r.get("distancia"))
        print("-" * 80)
else:
    print("Nenhum resultado encontrado.")


## 5 · Decisão D3 — Sistema de Memória do Agente: Escrita, Isolamento Seguro e LGPD
Para que o nosso agente Q se lembre das conversações passadas de forma inteligente e segura, precisamos implementar três contratos de função:
* `gravar(cliente_id, texto, quando_iso)`: Salva uma nova mensagem ou lembrete do agente.
* `buscar_memorias(cliente_id, query, n=3)`: Recupera as memórias mais relevantes daquele cliente.
* `esquecer(cliente_id)`: Apaga os dados de forma física e definitiva quando solicitado.

No código padrão de partida, a gravação e a busca acontecem **sem nenhum isolamento de segurança**, o que é um risco grave de vazamento de dados! Rodem a célula abaixo e vejam que, atualmente, um cliente consegue enxergar o histórico de outro. Vamos blindar essas funções com o parâmetro `cliente_id` e projetar a exclusão real física (`DELETE`) em conformidade com a LGPD.

In [9]:
from datetime import datetime
from kit_q.ingestao import ingerir_memorias

def gravar(cliente_id, texto, quando=None, origem=None):
    if quando is None:
        quando = datetime.now()
    v = emb([texto])[0]
    with conn.cursor() as cur:
        # ======================= D1/D3 · ESCRITA =======================
        # Vamos guardar o que o agente aprendeu com o cliente. Mas reparem: além de cliente_id,
        # texto e embedding, não deveríamos registrar o momento exato ("quando") da interação?
        # Sem isso não há recência (D4), não há retenção e não há trilha de auditoria.
        cur.execute("INSERT INTO memoria (cliente_id, texto, embedding, criado_em, origem) VALUES (%s, %s, %s, %s, %s)", (cliente_id, texto, v, quando, origem))
        # 🚨 e o timestamp (quando)? e a origem (seed vs. agente)?
        # ==============================================================

def buscar_memorias(cliente_id, query, n=3):
    if not cliente_id:
      return []
    v = emb([query])[0]
    with conn.cursor() as cur:
        # ======================= D3 · PRIVACIDADE E SEGURANÇA (ISOLAMENTO) =======================
        # Alerta de segurança máxima, turma! 🚨
        # Imaginem o Carlos (cli-002) perguntando sobre a renegociação da dívida dele e, por uma falha
        # nesta query, o sistema devolver o que a Ana (cli-003) negociou — ou o Bruno (cli-004)
        # "lembrar" da moto da Marina (cli-001). Isso não é um bug operacional: é uma violação
        # grave de privacidade, um incidente LGPD e o fim da confiança no agente.
        #
        # O contrato buscar_memorias recebe um cliente_id. Usem-no para blindar o SQL: o agente só
        # pode ler o histórico do cliente autenticado nesta conversa — nunca "o mais parecido" da base.
        # Dica extra do Prof (D4): entre memórias igualmente relevantes, como fazer as mais recentes
        # subirem? Pensem no ORDER BY combinando distância e idade da memória.
        cur.execute("SELECT texto FROM memoria WHERE cliente_id = %s ORDER BY embedding <=> %s, criado_em DESC LIMIT %s", (cliente_id, v, n))
        # =========================================================================================
        return [r[0] for r in cur.fetchall()]

def esquecer(cliente_id):
    # ======================= D3 · CONFORMIDADE COM A LGPD (HARD DELETE) =======================
    # Atenção, pessoal! Quando um cliente exerce o direito de exclusão (LGPD, art. 18), um
    # "soft delete" (marcar ativo = false e seguir a vida) NÃO atende à lei: o dado continua no disco,
    # continua em backup, continua "buscável" por quem errar um WHERE.
    #
    # Precisamos de um DELETE físico real, restrito ao cliente_id recebido, deixando o restante da base
    # intacto. E, como engenheiros(as) de dados responsáveis, registrem em log QUANDO e QUEM foi
    # apagado — a auditoria precisa provar a exclusão sem guardar o conteúdo excluído.
    if not cliente_id:
        return
    with conn.cursor() as cur:
        cur.execute("DELETE FROM memoria WHERE cliente_id = %s", (cliente_id,))
        conn.commit()
    # ==========================================================================================

with conn.cursor() as cur: cur.execute("TRUNCATE memoria")
ingerir_memorias(gravar)

# ==============================================================================
# INSPEÇÃO DIRETA DA TABELA MEMORIA
# ==============================================================================
# with conn.cursor() as cur:
#     # Selecionamos apenas as colunas de texto/metadados para não poluir a tela com vetores gigantes
#     cur.execute("SELECT id, cliente_id, texto, embedding, criado_em, origem FROM memoria LIMIT 10") # WHERE cliente_id = 'cli-001'
#     linhas = cur.fetchall()

#     print(f"Total de registros encontrados (máx 50): {len(linhas)}")
#     print("=" * 80)

#     if len(linhas) == 0:
#         print("🚨 A TABELA ESTÁ COMPLETAMENTE VAZIA!")
#         print("Certifique-se de que rodou a ingestão e executou 'conn.commit()' para salvar os dados.")
#     else:
#         for i, r in enumerate(linhas, 1):
#             print(f"Registro #{i}")
#             print(f"ID: {r[0]}")
#             print(f"Cliente ID: {r[1]}")
#             print(f"Texto (resumo): {r[2][:80]}...")
#             print(f"Origem: {r[3]}")
#             print(f"Criado em: {r[4]}")
#             print("-" * 50)

print("Bruno (cli-004) segunda via de cartão →", buscar_memorias("cli-004", "o que este cliente falou segunda via de cartão?", 2))
print("ANTES Marina (cli-001) aplicação em CDB →", buscar_memorias("cli-001", "Quanto o cliente aplicou em CDB?", 2))
#esquecer("cli-001")
print("DEPOIS Marina (cli-001) aplicação em CDB →", buscar_memorias("cli-001", "Quanto o cliente aplicou em CDB?", 2))

# RETORNO ORIGINAL DO PROFESSOR:
# 24 memórias gravadas via gravar() · 8 clientes
# Bruno (cli-004) pergunta sobre moto → ['Conversa curta: cliente atualizou o telefone de contato e comentou que a compra da moto ficou para o fim do ano.', 'Cliente contou que está juntando dinheiro para comprar uma moto e perguntou sobre crédito pessoal; renda informada de R$ 7.000; prefere pagar tudo por boleto, não gosta de débito automático.']
# Marina (cli-001) e o cartão → ['Cliente pediu aumento do limite do cartão Black; encaminhado para reanálise de crédito.', 'Cliente perguntou sobre o CDB Quantum e aplicou R$ 2.000 para a reserva de emergência; pediu para NÃO receber ofertas de cartão de crédito por enquanto.']

✅ 24 memórias gravadas via gravar() · 8 clientes
Bruno (cli-004) segunda via de cartão → ['Cliente pediu segunda via do cartão após perda; cartão anterior bloqueado; nova via isenta por ser a primeira ocorrência.', 'Cliente avisou viagem à Argentina e perguntou sobre uso do cartão no exterior e IOF; ativou o Quantum Global.']
ANTES Marina (cli-001) aplicação em CDB → ['Cliente perguntou sobre o CDB Quantum e aplicou R$ 2.000 para a reserva de emergência; pediu para NÃO receber ofertas de cartão de crédito por enquanto.', 'Cliente reclamou da tarifa de saque cobrada duas vezes no mês; foi orientada sobre os 2 saques gratuitos; ficou satisfeita com o estorno de uma tarifa.']
DEPOIS Marina (cli-001) aplicação em CDB → ['Cliente perguntou sobre o CDB Quantum e aplicou R$ 2.000 para a reserva de emergência; pediu para NÃO receber ofertas de cartão de crédito por enquanto.', 'Cliente reclamou da tarifa de saque cobrada duas vezes no mês; foi orientada sobre os 2 saques gratuitos; ficou satis

## 6 · ⚔️ Nosso Avaliador de Bancada (Rodem quantas vezes precisarem!)
Para nos ajudar a validar a solução de forma segura, criamos um avaliador que simula interações reais e testa nosso banco contra tentativas de vazamento de dados.

*Nota Pedagógica importante:* No último teste, o avaliador vai solicitar a exclusão de dados da cliente Ana (cli-003) para testar nossa função de conformidade com a LGPD. Por isso, caso vocês queiram rodar a avaliação de novo após a primeira execução, lembrem-se de limpar a tabela e rodar novamente a célula de ingestão das memórias (`TRUNCATE` + `ingerir_memorias`) para repopular os dados de teste!

In [ ]:
from avaliacao.avaliar_a1 import avaliar

# q_agente é opcional (check 10 = +5). Rode a seção 8 (Ollama + ChassiAgente) e
# reavalie com q_agente=q_agente para fechar OURO.
resultado = avaliar(
    conn, buscar_conhecimento, buscar_memorias, esquecer,
    q_agente=globals().get("q_agente"),  # None se ainda não rodou a seção 8
    grupo="GRUPO1",
)


## 7 · Evidências para a ficha (rode e deixe as saídas no notebook)

In [ ]:
# 7a · EXPLAIN da busca (o avaliador não mede isso; o Bloco B vai medir)
explain(conn, "SELECT id FROM conhecimento ORDER BY embedding <=> %s LIMIT 3", (emb(["tarifa"])[0],))
# 7b · teste de isolamento do Revisor(a): o Bruno tenta ler a Marina
print("Bruno vê:", buscar_memorias("cli-004", "moto da Marina", 3))
# 7c · esquecer() verificado: a Ana some do banco
esquecer("cli-003")
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM memoria WHERE cliente_id = 'cli-003'"); print("linhas da Ana após esquecer():", cur.fetchone()[0])

## 8 · Vamos Plugar Nosso Banco no Agente Q! (Ollama no Colab)
Este passo é super divertido! Aqui, vamos ver como a nossa infraestrutura de banco de dados serve como o sistema de suporte de decisão para o agente inteligente.

*Dica de Infraestrutura do Prof:* Para que a execução seja super veloz, prefiram configurar o seu ambiente no Google Colab para usar uma **GPU T4** (menu *Ambiente de Execução -> Alterar tipo de ambiente de execução -> T4 GPU*). Se rodarmos apenas em CPU, cada resposta do LLM pode levar cerca de 30 segundos. Como o modelo 3B é leve e compacto, às vezes ele pode tentar responder diretamente sem chamar a ferramenta na primeira tentativa, por isso colocamos um pequeno loop que tenta até 3 vezes para garantir a chamada!

In [ ]:
%%bash
set -e
apt-get install -y -qq zstd > /dev/null
command -v ollama > /dev/null || (curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1)
pgrep -x ollama > /dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
sleep 3
ollama pull qwen2.5:3b 2>&1 | tail -1

In [ ]:
%pip -q install ollama
from chassi_agente import ChassiAgente

def buscar_conhecimento_tool(query):
    '''Busca na base oficial da Quantum Finance (apenas status = 'oficial').'''
    rows = buscar_conhecimento(query, filtros={"status": "oficial", "data_min": "2025-08-13"}, k=3)   # 🚨 e a vigência? um FAQ oficial de 2025 ainda diz R$ 6,90…
    return "\n\n".join(f"[fonte: {r['doc_id']}] {r.get('titulo') or ''}\n{r['texto']}" for r in rows)

q_agente = ChassiAgente()
q_agente.registrar_ferramenta(buscar_conhecimento_tool,
    descricao="Busca políticas, tarifas e regras oficiais da Quantum Finance. Use para qualquer pergunta sobre a empresa.",
    parametros={"query": {"type": "string", "description": "termos de busca em português"}})

for tentativa in range(1, 4):                 # o 3B às vezes responde sem chamar a ferramenta
    q_agente.custo_chamadas_llm = 0
    resposta = q_agente.perguntar("Qual a tarifa de saque?")
    if q_agente.custo_chamadas_llm > 1:       # >1 chamada ao LLM = houve chamada de ferramenta
        break
print(resposta)

# reavalie passando o agente para ganhar o check 10 (lembre do TRUNCATE + ingerir_memorias antes):
# resultado = avaliar(conn, buscar_conhecimento, buscar_memorias, esquecer, q_agente=q_agente, grupo="...")

## 9 · Organização do Entregável do Grupo
Parabéns por chegar até aqui! O avaliador automático salvou um arquivo chamado `resultado_a1.json` na raiz do seu kit de laboratório.

Para concluir esta etapa, façam o download deste arquivo JSON, exportem este notebook com todas as saídas de execução salvas e preencham a seção §A1 da nossa `FICHA_DECISAO.md`. Compactem esses materiais em um único arquivo `.zip` com o nome do seu grupo (por exemplo, `grupoNN_operacaoQ.zip`) e realizem a entrega no portal oficial da FIAP!

In [ ]:
try:
    from google.colab import files
    files.download("resultado_a1.json")
except ImportError:                       # fora do Colab: o arquivo está na raiz do kit
    import os; print("resultado_a1.json em", os.path.abspath("resultado_a1.json"))